In [1]:
# ============================================================
# PHASE 2 — GRAPH CONSTRUCTION & BOTTLENECK AUDIT
# Run: python phase2.py
# ============================================================

import pickle
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Load Phase 1 ─────────────────────────────────────────────
with open('phase1_checkpoint.pkl', 'rb') as f:
    data = pickle.load(f)

df               = data['df']
trip_agg         = data['trip_agg']
THRESHOLD_SEVERE = data['THRESHOLD_SEVERE']
print(f"✅ Phase 1 loaded | df: {df.shape} | trips: {trip_agg.shape}")

# ── Corridor Stats ────────────────────────────────────────────
corridor_stats = df.groupby(
    ['source_center','destination_center','route_type']
).agg(
    median_delay_ratio =('delay_ratio',       'median'),
    mean_delay_ratio   =('delay_ratio',        'mean'),
    std_delay_ratio    =('delay_ratio',        'std'),
    trip_count         =('trip_uuid',          'nunique'),
    segment_count      =('delay_ratio',        'count'),
    sla_severe_rate    =('is_severe_delay',    'mean'),
    sla_extreme_rate   =('is_extreme_delay',   'mean'),
    avg_actual_time    =('segment_actual_time','mean'),
    avg_osrm_time      =('segment_osrm_time',  'mean'),
    avg_distance       =('segment_osrm_distance','mean'),
    src_state          =('src_state',          'first'),
    dst_state          =('dst_state',          'first'),
).reset_index()

corridor_stats['is_chronic']   = (
    (corridor_stats['median_delay_ratio'] > THRESHOLD_SEVERE) &
    (corridor_stats['trip_count'] >= 3)
).astype(int)
corridor_stats['is_interstate'] = (
    corridor_stats['src_state'] != corridor_stats['dst_state']
).astype(int)

print(f"Corridors: {len(corridor_stats)} | Chronic: {corridor_stats['is_chronic'].sum()}")

# ── Build Graph ───────────────────────────────────────────────
G = nx.DiGraph()
for _, row in corridor_stats.iterrows():
    src = str(row['source_center'])
    dst = str(row['destination_center'])
    if G.has_edge(src, dst):
        if row['median_delay_ratio'] > G[src][dst]['weight']:
            G[src][dst].update({
                'weight':          row['median_delay_ratio'],
                'sla_severe_rate': row['sla_severe_rate'],
                'trip_count':      row['trip_count'],
                'route_type':      row['route_type'],
                'is_chronic':      row['is_chronic'],
            })
    else:
        G.add_edge(src, dst,
                   weight          = row['median_delay_ratio'],
                   sla_severe_rate = row['sla_severe_rate'],
                   trip_count      = row['trip_count'],
                   route_type      = row['route_type'],
                   is_chronic      = row['is_chronic'])

print(f"Graph: {G.number_of_nodes()} nodes | {G.number_of_edges()} edges")

# ── Centrality ────────────────────────────────────────────────
print("Computing centrality (2-5 mins)...")
betweenness = nx.betweenness_centrality(G, weight='weight', normalized=True, k=200)
in_degree   = dict(G.in_degree())
out_degree  = dict(G.out_degree())
pagerank    = nx.pagerank(G, weight='weight', alpha=0.85)

hub_df = pd.DataFrame({
    'facility':   list(G.nodes()),
    'betweenness':[betweenness.get(n,0) for n in G.nodes()],
    'in_degree':  [in_degree.get(n,0)   for n in G.nodes()],
    'out_degree': [out_degree.get(n,0)  for n in G.nodes()],
    'pagerank':   [pagerank.get(n,0)    for n in G.nodes()],
})
hub_df['total_degree'] = hub_df['in_degree'] + hub_df['out_degree']

hub_incoming = (df.groupby('destination_center')['delay_ratio']
                .agg(avg_incoming_delay='mean', median_incoming_delay='median')
                .reset_index()
                .rename(columns={'destination_center':'facility'}))
hub_incoming['facility'] = hub_incoming['facility'].astype(str)

hub_outgoing = (df.groupby('source_center')['is_severe_delay']
                .mean().reset_index()
                .rename(columns={'source_center':'facility',
                                 'is_severe_delay':'outgoing_severe_rate'}))
hub_outgoing['facility'] = hub_outgoing['facility'].astype(str)

hub_volume = (pd.concat([
    df[['source_center','trip_uuid']].rename(columns={'source_center':'facility'}),
    df[['destination_center','trip_uuid']].rename(columns={'destination_center':'facility'})
]).groupby('facility')['trip_uuid'].nunique().reset_index(name='trip_volume'))
hub_volume['facility'] = hub_volume['facility'].astype(str)

hub_df = (hub_df
          .merge(hub_incoming, on='facility', how='left')
          .merge(hub_outgoing, on='facility', how='left')
          .merge(hub_volume,   on='facility', how='left')
          .fillna(0))

def normalize(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-8)

hub_df['bottleneck_score'] = (
    0.40 * normalize(hub_df['betweenness'])        +
    0.30 * normalize(hub_df['outgoing_severe_rate'])+
    0.20 * normalize(hub_df['trip_volume'])         +
    0.10 * normalize(hub_df['pagerank'])
)

hub_df['is_high_risk'] = (
    (hub_df['bottleneck_score']      > hub_df['bottleneck_score'].quantile(0.8)) &
    (hub_df['outgoing_severe_rate']  > hub_df['outgoing_severe_rate'].quantile(0.7)) &
    (hub_df['trip_volume']           >= 30)
).astype(int)

print(f"\nTop 5 bottleneck hubs:")
print(hub_df.nlargest(5,'bottleneck_score')[
    ['facility','bottleneck_score','outgoing_severe_rate','trip_volume']
].to_string())

# ── Save Checkpoint ───────────────────────────────────────────
with open('phase2_checkpoint.pkl', 'wb') as f:
    pickle.dump({
        'df':               df,
        'trip_agg':         trip_agg,
        'corridor_stats':   corridor_stats,
        'hub_df':           hub_df,
        'G':                G,
        'THRESHOLD_SEVERE': THRESHOLD_SEVERE,
    }, f, protocol=4)

print("\n✅ Phase 2 complete — saved phase2_checkpoint.pkl")

✅ Phase 1 loaded | df: (141661, 43) | trips: (14804, 30)
Corridors: 2804 | Chronic: 92
Graph: 1657 nodes | 2781 edges
Computing centrality (2-5 mins)...

Top 5 bottleneck hubs:
         facility  bottleneck_score  outgoing_severe_rate  trip_volume
25   IND000000ACB          0.720099              0.066997         1848
12   IND562132AAA          0.514591              0.055410         1366
628  IND400072AAJ          0.320399              1.000000           53
782  IND461001AAA          0.308047              1.000000            5
346  IND242001AAA          0.308011              1.000000            4

✅ Phase 2 complete — saved phase2_checkpoint.pkl
